## WRF Atölye

WRF detaylı dökümantasyon için [WRF Users Guide](https://www2.mmm.ucar.edu/wrf/users/wrf_users_guide/build/html/index.html)

### WPS (WRF Preproccessing System)

<img src="fig/wps_program_flow.png" width="600">

namelist.wps 4 kısımdan oluşuyor:
1. &share
2. &geogrid
3. &ungrib
4. &metgrid

---

&share kısımı simülasyonun başlangıç bitiş tarihlerini, zaman aralığını, kullanılacak WRF Modelini ve domain sayısını belirtir.
```fortran
&share
 wrf_core = 'ARW',
 max_dom = 2,
 start_date = '2019-09-04_12:00:00','2019-09-04_12:00:00',
 end_date   = '2019-09-06_00:00:00','2019-09-04_12:00:00',
 interval_seconds = 3600
/
```
interval_seconds, kullanacağımız verinin zaman aralığı. Burada 1 saat aralıklı ERA5 verileri kullanıldığı için 3600 olarak belirlenmiş.

---

Geogrid, model alanlarını tanımlayıp statik coğrafi verileri gridlere enterpole eder.

Verdiğimiz &geogrid sectiona göre fiziksel grid oluşturuyor.
```fortran
&geogrid
 parent_id         =   1,   1,
 parent_grid_ratio =   1,   3,
 i_parent_start    =   1,  53,
 j_parent_start    =   1,  25,
 e_we              =  150, 220,
 e_sn              =  130, 214,
 dx = 15000,
 dy = 15000,
 map_proj = 'lambert',
 ref_lat   =  33.00,
 ref_lon   = -79.00,
 truelat1  =  30.0,
 truelat2  =  60.0,
 stand_lon = -79.0,
 geog_data_res = 'default','default',
 geog_data_path = '/home/wrfuser/WPS_GEOG/'
/

<img src="fig/wps_nested.png" width="450">

***parent_id***, grid numarası

***parent_grid_ratio***, *(üstteki gridin çözünürlüğü) / (altaki gridin çözünürlüğü)*. Burada dış grid 15km çözünürlüğünde iken içerideki grid 5km çözünürlüğünde, içerideki grid için oran 3.

***i ve j_parent_start***, Domainin sol alt köşesinin x ve y kordinatlarını, bir dışarıdaki domainin sol alt köşesine göre veriyor.

***e_we e_sn***, Batı-Doğu ve Güney-Kuzey doğrultularındaki grid noktası sayısı.

***dx dy***, metre olarak çözünürlük

***geog_data_res***, griddeki arazinin detay seviyesini belirler. 'default' otomatik olarak en uygununu seçiyor. 

***geog_data_path***, coğrafi datanın konumu

#### Haritalama
Lambert Conformal Conic, havacılık haritalarında kullanılıyor ana özelliği lokal olarak açıları ve şekilleri koruması (conformal) ama uzaklıkları korumuyor. Açıları koruduğundan dolayı

<img src="fig/mapping/1.png" width="700">
<img src="fig/mapping/2.png" width="700">
<img src="fig/mapping/3.png" width="700">
<img src="fig/mapping/4.png" width="700">

***ref_lat ref_lon***, en dıştaki gridin orta noktasının kordinatları.

***truelat1 truelat2***, koninin Dünya'yı kestiği noktaların enlemleri, bu enlemlerde uzaklıkta bozulma yok.

***stand_lon***, haritanın oryantasyonunu belirtiyor.

##### geogrid.exe
```
./geogrid.exe >& geogrid.log
tail -f geogrid.log
```

---

ungrib, GRIB formatlı dosyadan meteorolojik veriyi çıkarır.

Kullandığımız data alma scripti pressure levels, single levels ve SST verilerini farklı dosyalara yazıyor, ERA5_PL ERA5_SFC ERA5_SST. Bu yüzden bunları ayrı ayrı ungrib yapmamız gerekiyor.

&ungrib kısmında prefix kısmına sırayla (hangi sırada olduğunun önemi yok) ERA5_PL ERA5_SFC ERA5_SST yazarak hepsini ungrib yapıyoruz.

```fortran
&ungrib
 out_format = 'WPS',
 prefix = 'ERA5_PL',
/
```

##### ungrib.exe
ungrib.exe'yi çalıştırmadan önce ```./link_grib.csh (data.py ile aldığımız dosyaların yolu)``` ile data dosyalarını buraya "kopyalıyoruz".

Daha sonra GRIB dosyasını ungrib yaparken verilerin GRIB içinde nasıl bulunduğunu belirtmek için ```ln -sf ungrib/Variable_Tables/Vtable.ECMWF Vtable``` ile Vtable oluşturuyoruz.

Son olarak ```./ungrib.exe```'yi çalıştırıyoruz, bu örnekte ERA5_PL için çalıştırdıktan sonra namelist.wps içinde prefixe sırayla ERA5_SFC ve ERA5_SST'yi de yazarak çalıştırıyoruz.

---

metgrid, ungrib tarafından çıkarılan meteorolojik veriyi, geogrid tarafından oluşturulan girdlere yatay olarak enterpole eder.
```fortran
&metgrid
 fg_name = 'ERA5_SFC','ERA5_SST','ERA5_PL',
/
```
##### metgrid.exe
Metgrid, ungrib ve geogrid'e göre daha ağır bir işlem olduğu için paralel olarak çalıştırmak daha hızlı sonuç verir.
```nohup mpirun -np 48 ./metgrid.exe < /dev/null > metgrid.log 2>&1 &```
nohup, server'da çalışırken exit ile disconnect çektiğimiz zaman işlemin çalışmaya devam etmesini sağlar.

mpirun, paralelliği sağlayan program. -np ile çekirdek sayısını belirtiyoruz.

'> metgrid.log 2>&1 &, ile metgrid.exe'nin çıktısını metgrid.log dosyasına yazıyoruz daha sonra ```tail -f metgrid.log``` ile çıktıyı izleyebiliriz. Bu disconnect çekeceğimiz zaman veya yanlışlıkla programı durdurmamak için kullanılır.


***

WPS geogrid oluşturmak için site [WRF Domain Wizard](https://jiririchter.github.io/WRFDomainWizard/)

---

### WRF

##### &time_control
Zaman ve çıktıların nasıl oluşturulacağının belirlendiği yer
```fortran
&time_control
 run_days                            = 0,
 run_hours                           = 3,
 run_minutes                         = 0,
 run_seconds                         = 0,
 start_year                          = 2026,
 start_month                         = 01,
 start_day                           = 11,
 start_hour                          = 00,
 end_year                            = 2026,
 end_month                           = 01,
 end_day                             = 11,
 end_hour                            = 03,
 interval_seconds                    = 3600,
```

```fortran
 write_hist_at_0h_rst                = .true.,
 override_restart_timers             = .true.,   wrfrst zamanı yerine namelist.input dakini kullan
 adjust_output_times                 = .true.,   zaman kaymalarını engelleme
```

```fortran
 input_from_file                     = .true.,
 history_interval                    = 60,
 frames_per_outfile                  = 1,
 io_form_history                     = 2,
 io_form_restart                     = 2,
 io_form_input                       = 2,
 io_form_boundary                    = 2,
 iofields_filename                   = "variables.txt",
 ignore_iofields_warning             = .true.,
 debug_level                         = 0,
 restart                             = .false.,
```

***history_interval***, simülasyon çıktısının hangi süre aralığında dosyaya yazılacağını belirtir, dakika.

***frames_per_outfile***, bir simülasyon çıktı dosyası içine kaç tane zaman adımının yazılacağını belirtir.

***io_form_**** history, restart, input ve boundary doyalarının netCDF formatıyla yazılmasını belirtir.

***iofields_filename***, çıktıların nasıl olacağını belirten dosya

---

variables.txt
```
+:h:7:AFWA_TOTPREC,AFWA_RAIN,AFWA_SNOW,AFWA_SNOWFALL,AFWA_ICE,AFWA_FZRA,AFWA_PWAT,AFWA_VIL,TCOLI_MAX
-:h:0:AFWA_TOTPREC,AFWA_RAIN,AFWA_SNOW,AFWA_SNOWFALL,AFWA_ICE,AFWA_FZRA,AFWA_PWAT,AFWA_VIL,TCOLI_MAX

+:h:3:XLAT,XLONG,XTIME

+:h:7:XLAT,XLONG,XTIME,AFWA_MSLP,TSK,RAINC,RAINNC,RAINSH,SNOWNC,HAILNC,GRAUPELNC,SNOW,SNOWH,T2,Q2,U10,V10,SWDOWN,OLR,QFX,SFCEVP,SFROFF,UDROFF,SSTSK
```
---

```fortran
 auxhist7_outname                    = "wrfsfc_d<domain>_<date>",
 auxhist7_interval                   = 5,
 frames_per_auxhist7                 = 1,
 io_form_auxhist7                    = 2,
```
```fortran
 auxinput4_inname                    = "wrflowinp_d<domain>",
 auxinput4_interval                  = 60,
 io_form_auxinput4                   = 2,
```

---


##### &domains

```fortran
 &domains
 max_dom                             = 1,
 time_step                           = 5,
 time_step_fract_num                 = 0,
 time_step_fract_den                 = 1,

 use_adaptive_time_step              = .true.,
 target_cfl                          = 1.2,
 max_step_increase_pct               = 5,
 starting_time_step                  = -1,
 max_time_step                       = -1,
 min_time_step                       = -1,
 adaptation_domain                   = 1,
 step_to_output_time                 = .true.,
```

***max_dom***, domain sayısı

***time_step _fract_num fract_den***, simülasyonun çalışma aralığı, tam sayı olmayan zaman adımlarının kullanılmasını sağlıyor.

***use_adaptive_time_step***, simülasyon doğruluğunu bozmayacak kararlı anlarda, time_step'i arttırıp simülasyonu hızlandırıyor. 

***target_cfl***, Courant–Friedrichs–Lewy condition (_cfl dikey, _hcfl yatay). time_step'in doğruluğu bozmayacak şekilde artabileceği maksimum değeri belirliyor. Detaylı bilgi için [wiki](https://en.wikipedia.org/wiki/Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition) ve [blog](https://www.simscale.com/blog/cfl-condition/).

***max_step_increase_pct***, anlık cfl target_cfl'den küçükse zaman adımının maksimum ne kadar artabileceğini belirtiyor, %5. 

***starting_time_step max_time_step min_time_step***, zaman adımlarının başlangıç, minimum ve maksimum değerlerini belirliyor. -1 otomatik belirlenmesini sağlıyor.

***adaptation_domain***, adaptive time step'in hangi domainin cfl değerine göre belirleneceğini gösteriyor. Tek domainde 1.

***step_to_output_time***, değişen zaman adımlarının istediğimiz aralıklarda olmasını sağlıyor.


```fortran
 e_we                                = 900,
 e_sn                                = 800,
 e_vert                              = 45,
 p_top_requested                     = 10000,
 num_metgrid_levels                  = 38,
 num_metgrid_soil_levels             = 4,
 dx                                  = 3000,
 dy                                  = 3000,
 grid_id                             = 1,
 parent_id                           = 0,
 i_parent_start                      = 1,
 j_parent_start                      = 1,
 parent_grid_ratio                   = 1,
 parent_time_step_ratio              = 1,
 feedback                            = 1,
 smooth_option                       = 0,
 /
```

***e_we e_sn e_vert***, Batı-Doğu, Güney-Kuzey ve dikey doğrultulardaki grid noktası sayısı.

***p_top_requested***, maksimjum atmosfer yüksekliği, Pa. 10000Pa = 100hPa

***num_metgrid_levels num_metgrid_soil_levels***, girdi verisindeki atmosferik ve topraktaki dikey level, e_vert bundan büyükse interpole edilir.

***grid_id parent_id***, grid no ve onun dışındaki giridin nosu. Tek gridde dışarıda grid olmadığı için parent_grid = 0.

***feedback***, içerideki yüksek çözünürlüklü domainin, dışarıdakini beslemesi.

***smooth_option***, geri beslemenin iki domain sınırdaki smoothlaştırma algoritması kullanımı.

---

#### &bdy_control
```fortran
&bdy_control
 spec_bdy_width                      = 11,
 spec_zone                           = 1,
 relax_zone                          = 10,
 spec_exp                            = 0.33,
 specified                           = .true.,
 nested                              = .false.,
 /
```

***spec_zone***, en dışta hesaplamanın yapılmayacağı, doğrudan input verilerinin kullanılacağı grid genişliği

***relax_zone***, input verilerinin ve WRF simülasyon sonuçlarının kombine edildiği grid genişliği. Dışarıdan içeri gittikçe input verisinin yüzdesi azalırken WRF'ınki artar.

***spec_bdy_width***, spec_zone + relax_zone

***spec_exp***, relax_zone dan içeriye giderken input verisinin yüzdesinin azalma miktarı, exponansiyel.

***specified***, sınır verilerinin input olarak alınacağını belirtir.

***nested***, sınır verilerinin bir üst domainden (parent_domain) alınıp alınmayacağı. Nested domain ile çalışırken .true. 

---

##### real.exe wrf.exe

real.exe, met_em.* dosyalarını initialize edip wrf.exe için uygun hale getirir.

wrf.exe, ana simülasyon

```cd ../WRF/test/em_real```

```ln -sf ../../../WPS/met_em*``` , WPS metgridler "kopyalanır".

```nohup mpirun -np 8 ./real.exe < /dev/null > real.log 2>&1 &``` , 8 çekirdekte real.exe çalıştırma

```tail rsl.error.0000``` , real.exe takibi

```nohup mpirun -np 64 ./wrf.exe < /dev/null > simulation_output.log 2>&1```, 64 çekirdekte wrf.exe çalıştırma

```tail rsl.error.0000``` , wrf.exe takibi